In [1]:
!pip install transformers fsspec datasets

In [4]:
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments



In [5]:
from datasets import load_dataset

ds = load_dataset("JosefGoldstein/aimlessinnovations_customer_sentiment_v2")



README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/25032 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3129 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3129 [00:00<?, ? examples/s]

In [6]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label_text', 'label'],
        num_rows: 25032
    })
    validation: Dataset({
        features: ['id', 'text', 'label_text', 'label'],
        num_rows: 3129
    })
    test: Dataset({
        features: ['id', 'text', 'label_text', 'label'],
        num_rows: 3129
    })
})

In [7]:
train_dataset = ds["train"].shuffle(seed=42).select(range(2000))

test_dataset = ds["test"].shuffle(seed=42).select(range(500))

validation_dataset = ds["validation"].shuffle(seed=42).select(range(500))

In [8]:
print(set(train_dataset["label_text"]))

{'very_negative', 'very_positive', 'karen', 'negative', 'positive', 'neutral'}


In [9]:
from collections import Counter

print(Counter(train_dataset['label']))

Counter({5: 384, 1: 362, 4: 339, 2: 328, 0: 304, 3: 283})


In [10]:
tokenizer=BertTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
def tokenize_fn(example):
  return tokenizer(example["text"],truncation=True,padding="max_length",max_length=256)

In [12]:

# Apply tokenization + rename + format in a single flow
def preprocess(ds):
    ds = ds.map(tokenize_fn, batched=True, remove_columns=["text"])  # remove raw text (saves memory)
    ds = ds.rename_column("label", "labels")
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds


In [13]:
train_dataset = preprocess(train_dataset)
test_dataset = preprocess(test_dataset)
validation_dataset=preprocess(validation_dataset)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [14]:
train_dataset[0]

{'labels': tensor(0),
 'input_ids': tensor([ 101, 1045, 2123, 1521, 1056, 3305, 2339, 1045, 2001, 5338, 2005, 7829,
          999, 1045, 1521, 2310, 2985, 5190, 2182,  999, 8081, 2023,  999,  999,
          999,  102,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,

In [20]:
id2label = {0:"karen", 1:"very_negative", 2: "negative",3:"neutral",4:"positive",5:"very_positive"}
label2id = {v:i for i,v in id2label.items()}

In [21]:
label2id

{'karen': 0,
 'very_negative': 1,
 'negative': 2,
 'neutral': 3,
 'positive': 4,
 'very_positive': 5}

In [22]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=6,
    id2label=id2label,
    label2id=label2id
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [23]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./bert-finetuned-customer_sentiment",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    #logging_dir="./logs",
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none"
)

In [24]:

# 5. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
)

In [26]:
trainer.train()

Step,Training Loss
500,0.303748


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.3037481079101563, metrics={'train_runtime': 198.0158, 'train_samples_per_second': 20.2, 'train_steps_per_second': 2.525, 'total_flos': 526241009664000.0, 'train_loss': 0.3037481079101563, 'epoch': 2.0})

In [27]:
trainer.save_model("./bert-finetuned-customer-sentiment")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [28]:
tokenizer.save_pretrained("./bert-finetuned-customer-sentiment")

('./bert-finetuned-customer-sentiment/tokenizer_config.json',
 './bert-finetuned-customer-sentiment/tokenizer.json')

In [29]:

# 7. Evaluate
metrics = trainer.evaluate()

In [30]:
print(metrics)

{'eval_loss': 0.03222110867500305, 'eval_runtime': 7.8574, 'eval_samples_per_second': 63.634, 'eval_steps_per_second': 8.018, 'epoch': 2.0}


In [31]:
tokenizer = BertTokenizer.from_pretrained("/content/bert-finetuned-customer-sentiment")
model = BertForSequenceClassification.from_pretrained("/content/bert-finetuned-customer-sentiment")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [32]:

from transformers import pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

In [34]:
# Predict
text ="it's too amazing I really loved it i can't take my eyes off of it"
result = classifier(text)

In [35]:

result

[{'label': 'very_positive', 'score': 0.787899374961853}]

In [36]:
results = trainer.evaluate(test_dataset)
print(results)

{'eval_loss': 0.04204043745994568, 'eval_runtime': 8.2416, 'eval_samples_per_second': 60.668, 'eval_steps_per_second': 7.644, 'epoch': 2.0}


In [37]:

from huggingface_hub import notebook_login
notebook_login()


In [38]:

from huggingface_hub import whoami
#print(whoami())

In [39]:
tokenizer.push_to_hub("Basma21/my-bert-customer-sentiment")

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Basma21/my-bert-customer-sentiment/commit/107ba0f14e20d2bd6724911dbf9e4314e3b451fe', commit_message='Upload tokenizer', commit_description='', oid='107ba0f14e20d2bd6724911dbf9e4314e3b451fe', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Basma21/my-bert-customer-sentiment', endpoint='https://huggingface.co', repo_type='model', repo_id='Basma21/my-bert-customer-sentiment'), pr_revision=None, pr_num=None)

In [41]:
trainer.push_to_hub("Basma21/my-bert-customer-sentiment")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ntiment/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...ntiment/model.safetensors:  35%|###4      |  152MB /  438MB            

CommitInfo(commit_url='https://huggingface.co/Basma21/bert-finetuned-customer_sentiment/commit/0ed0ce129741cfc38441df568c8ce2725c8df933', commit_message='Basma21/my-bert-customer-sentiment', commit_description='', oid='0ed0ce129741cfc38441df568c8ce2725c8df933', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Basma21/bert-finetuned-customer_sentiment', endpoint='https://huggingface.co', repo_type='model', repo_id='Basma21/bert-finetuned-customer_sentiment'), pr_revision=None, pr_num=None)

In [42]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("Basma21/bert-finetuned-customer_sentiment")
model = AutoModelForSequenceClassification.from_pretrained("Basma21/bert-finetuned-customer_sentiment")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [43]:
text = "i hate it"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

outputs = model(**inputs)

In [44]:
import torch

pred = torch.argmax(outputs.logits, dim=-1)

print(pred)

tensor([2])


In [45]:
model.config.id2label[pred.item()]

'negative'